# Pick & Go — Entraînement YOLOv8 (Google Colab)

**Instructions :**
1. `Exécution > Modifier le type d'exécution` → choisir **GPU T4**
2. Exécuter toutes les cellules dans l'ordre
3. Télécharger `best.pt` à la fin

> ⚠️ Si une cellule plante, relancez-la seule — inutile de tout relancer.

In [ ]:
# 0. Vérification GPU + espace disque
import subprocess, os

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                      '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() if gpu.returncode == 0 else 'Aucun GPU detecte!')

disk = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
print('Disque:', disk.stdout.split('\n')[1])

import psutil
ram = psutil.virtual_memory()
print(f'RAM: {ram.available/1e9:.1f} GB disponible / {ram.total/1e9:.1f} GB total')

In [ ]:
# 1. Installation des dépendances
!pip install ultralytics icrawler -q
import ultralytics
print('Ultralytics:', ultralytics.__version__)

In [ ]:
# 2. Catalogue des produits
PRODUCTS = {
    'Bouteille_Eau': {'nom': "Bouteille d'Eau", 'prix': 300},
    'Cahier':        {'nom': 'Cahier',           'prix': 500},
    'Stylo':         {'nom': 'Stylo',            'prix': 200},
    'Gazelle':       {'nom': 'Biere Gazelle',    'prix': 800},
    'Choco_Pain':    {'nom': 'Choco Pain',       'prix': 250},
    'Bissap':        {'nom': 'Jus Bissap',       'prix': 400},
    'Riz':           {'nom': 'Sac de Riz',       'prix': 1500},
    'Huile':         {'nom': 'Huile',            'prix': 2000},
}

SEARCH_QUERIES = {
    'Bouteille_Eau': 'bouteille eau minerale plastique',
    'Cahier':        'cahier scolaire spirale',
    'Stylo':         'stylo bille bleu noir',
    'Gazelle':       'gazelle biere bouteille senegal',
    'Choco_Pain':    'pain chocolat viennoiserie',
    'Bissap':        'bissap jus hibiscus sachet',
    'Riz':           'sac riz 5kg senegal',
    'Huile':         'huile cuisine bouteille litre',
}

IMAGES_PER_CLASS = 80
TRAIN_RATIO = 0.8
print(f'Catalogue: {len(PRODUCTS)} classes')
for k, v in PRODUCTS.items():
    print(f'  {k}: {v["prix"]} FCFA')

In [ ]:
# 3. Collecte des images
import os, shutil, random
from pathlib import Path
from icrawler.builtin import BingImageCrawler

RAW = Path('data/raw')

for class_name in PRODUCTS:
    save_dir = RAW / class_name
    save_dir.mkdir(parents=True, exist_ok=True)
    existing = list(save_dir.glob('*.*'))
    if len(existing) >= IMAGES_PER_CLASS:
        print(f'[OK] {class_name}: {len(existing)} images deja presentes.')
        continue
    need = IMAGES_PER_CLASS - len(existing)
    print(f'[DL] {class_name}: telechargement de {need} images...')
    try:
        crawler = BingImageCrawler(
            feeder_threads=1, parser_threads=1, downloader_threads=2,
            storage={'root_dir': str(save_dir)}
        )
        crawler.crawl(keyword=SEARCH_QUERIES[class_name],
                      max_num=IMAGES_PER_CLASS, min_size=(80, 80))
    except Exception as e:
        print(f'  Erreur: {e}')
    count = len(list(save_dir.glob('*.*')))
    print(f'  -> {count} images')

print('\nResume collecte:')
for class_name in PRODUCTS:
    n = len(list((RAW / class_name).glob('*.*')))
    status = 'OK' if n >= 20 else 'INSUFFISANT'
    print(f'  [{status}] {class_name}: {n} images')

In [ ]:
# 4. Annotation + split train/val
from pathlib import Path
import shutil, random

VALID_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

for dest in ['data/train/images', 'data/train/labels',
             'data/val/images',   'data/val/labels']:
    Path(dest).mkdir(parents=True, exist_ok=True)

class_names = list(PRODUCTS.keys())
total_train = total_val = 0

for class_id, class_name in enumerate(class_names):
    raw_dir = Path('data/raw') / class_name
    images = [f for f in raw_dir.iterdir() if f.suffix.lower() in VALID_EXT]
    if not images:
        print(f'  SKIP {class_name}: aucune image')
        continue
    random.shuffle(images)
    split = max(1, int(len(images) * TRAIN_RATIO))
    for phase, subset in [('train', images[:split]), ('val', images[split:])]:
        for img in subset:
            stem = f'{class_name}_{img.stem}'
            dst_img = f'data/{phase}/images/{stem}{img.suffix}'
            dst_lbl = f'data/{phase}/labels/{stem}.txt'
            shutil.copy2(img, dst_img)
            Path(dst_lbl).write_text(f'{class_id} 0.5 0.5 0.9 0.9\n')
    t, v = split, len(images) - split
    print(f'  {class_name:20s}: {t} train | {v} val')
    total_train += t; total_val += v

print(f'\nDataset: {total_train} train | {total_val} val')

In [ ]:
# 5. Création du data.yaml
yaml_lines = [
    'path: /content',
    'train: data/train/images',
    'val:   data/val/images',
    '',
    f'nc: {len(PRODUCTS)}',
    'names:',
]
for name in PRODUCTS:
    yaml_lines.append(f'  - {name}')

yaml_content = '\n'.join(yaml_lines) + '\n'
with open('data.yaml', 'w') as f:
    f.write(yaml_content)
print(yaml_content)

In [ ]:
# 6. Entraînement YOLOv8 (GPU T4)
import torch
from ultralytics import YOLO

print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

model = YOLO('yolov8n.pt')  # nano = plus leger, parfait pour Colab

results = model.train(
    data='data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,        # diminuer a 8 si OOM
    device=0 if torch.cuda.is_available() else 'cpu',
    project='models',
    name='pick_and_go',
    exist_ok=True,
    patience=10,     # arret si pas d'amelioration
    verbose=True,
)

print('\nEntrainement termine!')
print('Metriques finales:')
print(f'  mAP50:   {results.results_dict.get("metrics/mAP50(B)", "N/A")}')
print(f'  mAP50-95:{results.results_dict.get("metrics/mAP50-95(B)", "N/A")}')

In [ ]:
# 7. Vérification + localisation du modèle
import subprocess
from pathlib import Path

# Cherche best.pt partout dans le dossier courant
result = subprocess.run(['find', '.', '-name', 'best.pt'], capture_output=True, text=True)
found = result.stdout.strip().split('\n')
found = [f for f in found if f]

if found:
    print('Modele(s) trouve(s):')
    for f in found:
        size = Path(f).stat().st_size / 1e6
        print(f'  {f}  ({size:.1f} MB)')
    MODEL_PATH = found[0]
    print(f'\nUtilisation de: {MODEL_PATH}')
else:
    print('ERREUR: best.pt introuvable!')
    print('Verifiez que la cellule 6 a bien termine sans erreur.')
    MODEL_PATH = None

In [ ]:
# 8. Test rapide du modèle
if MODEL_PATH:
    from ultralytics import YOLO
    from pathlib import Path
    import glob

    m = YOLO(MODEL_PATH)
    print('Classes:', list(m.names.values()))

    # Prendre une image de val pour tester
    test_imgs = glob.glob('data/val/images/*')[:3]
    if test_imgs:
        for img in test_imgs:
            r = m(img, conf=0.3, verbose=False)[0]
            dets = [(m.names[int(b.cls)], float(b.conf)) for b in r.boxes]
            print(f'  {Path(img).name}: {dets if dets else "rien detecte"}')
    else:
        print('Pas dimage de validation disponible.')
else:
    print('Skipped: pas de modele trouve.')

In [ ]:
# 9. Téléchargement du modèle
from google.colab import files
from pathlib import Path

if MODEL_PATH and Path(MODEL_PATH).exists():
    print(f'Telechargement de {MODEL_PATH}...')
    files.download(MODEL_PATH)
    print()
    print('Une fois telecharge, place best.pt ici dans le repo:')
    print('  PickAndGo/models/pick_and_go/weights/best.pt')
else:
    print('ERREUR: Aucun modele a telecharger.')
    print('Relance la cellule 6 (entrainement) puis la cellule 7 (verification).')